In [0]:
%run "/Workspace/Users/wade.amit8895@outlook.com/Azure-Databricks-Projects/customer_billing/notebooks/utility/common_functions"

In [0]:
dbutils.widgets.text("pipeline_name", "customer_billing_pipeline")
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("pipeline_start_time", "")
dbutils.widgets.text("log_table", "control.etl_run_log")
log_table = dbutils.widgets.get("log_table")

pipeline_name = dbutils.widgets.get("pipeline_name")
task_name = "emai_notification"
run_id = dbutils.widgets.get("run_id")
pipeline_run_id = run_id
pipeline_start_time = dbutils.widgets.get("pipeline_start_time")

In [0]:

if not log_table or not str(log_table).strip():
    log_table = "control.etl_run_log"
# Read the actual log table schema supplied for this solution.
log_df = spark.table(log_table)
required_columns = {"run_id", "pipeline_name", "task_name", "file_name", "table_name","layer", "business_date", "run_start_time", "run_end_time", "run_status", "total_inserted_count", 
"total_updated_count","error_message", "email_status"}
missing_columns = sorted(required_columns - set(log_df.columns))

if missing_columns:
    raise ValueError(f"{log_table} is missing expected columns: {', '.join(missing_columns)}")

run_df = log_df.filter((F.col("pipeline_name") == pipeline_name) & 
(F.to_timestamp(F.col("run_start_time")) >= F.to_timestamp(F.lit(pipeline_start_time))))
records = run_df.orderBy("run_start_time", "layer", "task_name").collect()

if not records:
    raise RuntimeError(f"No records found for pipeline={pipeline_name} " f"since {pipeline_start_time}.")
print(f"Found {len(records)} log record(s) " f"for pipeline {pipeline_name}.")

In [0]:
def _text(value, default="N/A"):
    if value is None:
        return default
    value = str(value).strip()
    return value if value else default


def _status_category(value):
    status = _text(value, "").lower()
    if any(word in status for word in ("fail", "error", "exception")):
        return "failed"
    if any(word in status for word in ("success", "succeeded", "complete", "completed")):
        return "success"
    if any(word in status for word in ("running", "in progress", "started")):
        return "running"
    if any(word in status for word in ("cancel", "abort", "skip")):
        return "other"
    return "other"


def _format_datetime(value):
    if value is None:
        return "N/A"
    try:
        return value.strftime("%Y-%m-%d %H:%M:%S")
    except Exception:
        return str(value)

# Aggregate activity-level results.
total_tasks = len(records)
success_count = 0
failed_count = 0
running_count = 0
other_count = 0
total_inserted = 0
total_updated = 0

layer_summary = {}
failed_details = []
activity_details = []
start_times = []
end_times = []

for row in records:
    status_raw = row["run_status"]
    category = _status_category(status_raw)
    layer = _text(row["layer"], "Unknown")

    if category == "success":
        success_count += 1
    elif category == "failed":
        failed_count += 1
    elif category == "running":
        running_count += 1
    else:
        other_count += 1

    inserted = int(row["total_inserted_count"] or 0)
    updated = int(row["total_updated_count"] or 0)
    total_inserted += inserted
    total_updated += updated

    if layer not in layer_summary:
        layer_summary[layer] = {"total": 0, "success": 0, "failed": 0, "running": 0, "other": 0, "inserted": 0, "updated": 0}

    layer_summary[layer]["total"] += 1
    layer_summary[layer][category] += 1
    layer_summary[layer]["inserted"] += inserted
    layer_summary[layer]["updated"] += updated

    start_value = row["run_start_time"]
    end_value = row["run_end_time"]

    if start_value is not None:
        start_times.append(start_value)
    if end_value is not None:
        end_times.append(end_value)

    detail = {
        "layer": layer,
        "task": _text(row["task_name"]),
        "file": _text(row["file_name"]),
        "table": _text(row["table_name"]),
        "status": _text(status_raw),
        "start": _format_datetime(start_value),
        "end": _format_datetime(end_value),
        "inserted": inserted,
        "updated": updated,
        "error": _text(row["error_message"], "")
    }
    activity_details.append(detail)

    if category == "failed":
        failed_details.append(detail)

# Pipeline start parameter takes precedence over activity start times.
overall_start = min(start_times) if start_times else None
overall_end = max(end_times) if end_times else None

if pipeline_start_time:
    try:
        parsed_start = datetime.fromisoformat(
            str(pipeline_start_time).replace("Z", "+00:00")
        )
        overall_start = parsed_start
    except Exception:
        print(
            "Warning: pipeline_start_time could not be parsed; "
            "using earliest run_start_time from the log table."
        )

if overall_start and overall_end:
    # Normalize timezone awareness to avoid subtracting aware from naive datetimes.
    if getattr(overall_start, "tzinfo", None) is not None and getattr(overall_end, "tzinfo", None) is None:
        overall_start = overall_start.replace(tzinfo=None)
    elif getattr(overall_start, "tzinfo", None) is None and getattr(overall_end, "tzinfo", None) is not None:
        overall_end = overall_end.replace(tzinfo=None)

    duration_seconds = max(0, (overall_end - overall_start).total_seconds())
    hours, remainder = divmod(int(duration_seconds), 3600)
    minutes, seconds = divmod(remainder, 60)
    duration_text = f"{hours:02d}:{minutes:02d}:{seconds:02d}"
else:
    duration_text = "Not available"

if failed_count:
    overall_status = "FAILED"
elif running_count or other_count:
    overall_status = "COMPLETED WITH NON-SUCCESS TASKS"
else:
    overall_status = "SUCCESS"

print(f"Overall status: {overall_status}")
print(f"Tasks: {total_tasks}; Success: {success_count}; Failed: {failed_count}")
print(f"Inserted: {total_inserted}; Updated: {total_updated}")


In [0]:
# Build a plain-text message. The shared send_mail_fn applies the common HTML template.
subject = f"Pipeline Execution Report | {pipeline_name}"

def esc(value):
    return escape(str(value if value is not None else ""))

def td(value):
    return f'<td style="padding:9px 8px;border:1px solid #cbd5e1;vertical-align:top;">{value}</td>'

def section(title, subtitle):
    return f'<div style="border-left:5px solid #2864c5;padding:7px 12px;margin:26px 0 12px;"><h3 style="margin:0;color:#202b65;">{esc(title)}</h3><p style="margin:5px 0 0;color:#667085;font-size:11px;">{esc(subtitle)}</p></div>'

def badge(value):
    category = _status_category(value)
    colors = {"success":("#e2f5eb","#008b57"),"failed":("#fde8e8","#dc3545"),"running":("#fff3cd","#a66b00"),"other":("#e9edf3","#526071")}
    bg, fg = colors[category]
    return f'<span style="background:{bg};color:{fg};padding:5px 10px;border-radius:16px;font-weight:bold;">{esc(value)}</span>'

table_css = "width:100%;border-collapse:collapse;font:11px Arial,sans-serif;color:#24324b;"
th_css = "padding:9px 8px;border:1px solid #cbd5e1;text-align:left;background:#353c85;color:#fff;"

overview = [
    ("Pipeline Name", pipeline_name), ("Pipeline Run ID", pipeline_run_id),
    ("Overall Status", overall_status), ("Execution Start", _format_datetime(overall_start)),
    ("Execution End", _format_datetime(overall_end)), ("Total Duration", duration_text)
]
overview_html = "".join(
    f"<tr>{td(esc(k))}<td style='padding:9px 8px;border:1px solid #cbd5e1;background:#f0f4fb;'>{esc(v)}</td></tr>"
    for k, v in overview
)

kpis = [
    ("TOTAL ACTIVITIES", total_tasks, "#eaf2ff", "#2864c5"),
    ("SUCCESSFUL", success_count, "#e2f5eb", "#009b65"),
    ("FAILED", failed_count, "#fdecec", "#dc3545"),
    ("INSERTED", f"{total_inserted:,}", "#f0eaff", "#8050d5"),
    ("UPDATED", f"{total_updated:,}", "#fff3dc", "#c17b13")
]
kpi_html = "".join(
    f"<td style='width:20%;padding:13px 4px;text-align:center;background:{bg};border:1px solid #d9e2f0;'><div style='font-size:24px;font-weight:bold;color:{color};'>{value}</div><div style='font-size:10px;font-weight:bold;color:{color};'>{label}</div></td>"
    for label, value, bg, color in kpis
)

layer_html = ""
for layer in sorted(layer_summary):
    x = layer_summary[layer]
    layer_html += "<tr>" + "".join(td(v) for v in [
        esc(layer), x["total"], x["success"], x["failed"], x["running"],
        x["other"], f'{x["inserted"]:,}', f'{x["updated"]:,}'
    ]) + "</tr>"

activity_html = ""
for i, x in enumerate(activity_details, 1):
    vals = [i, esc(x["layer"]), esc(x["task"]), esc(x["file"]), esc(x["table"]),
            badge(x["status"]), esc(x["start"]), esc(x["end"]),
            f'{x["inserted"]:,}', f'{x["updated"]:,}', esc(x["error"] or "-")]
    bg = "#ffffff" if i % 2 else "#f0f4fb"
    activity_html += f"<tr style='background:{bg};'>" + "".join(td(v) for v in vals) + "</tr>"

if failed_details:
    failed_html = ""
    for x in failed_details:
        failed_html += "<tr>" + "".join(td(esc(v)) for v in [
            x["layer"], x["task"], x["file"], x["table"],
            x["error"] or "No error recorded"
        ]) + "</tr>"
    failed_section = section("Failed Activity Details", "Activities that failed") + f"""
    <table style="{table_css}">
    <tr>{''.join(f'<th style="{th_css}">{h}</th>' for h in ["Layer","Task","File","Table","Error Details"])}</tr>
    {failed_html}</table>"""
else:
    failed_section = section("Failed Activity Details", "No failed activities were recorded") + """
    <div style="padding:14px;background:#e2f5eb;color:#008b57;border:1px solid #a7dfc2;">
    &#10003; No failed activities recorded for this execution.</div>"""

status_bg = "#fde8e8" if overall_status == "FAILED" else "#e2f5eb"
status_color = "#dc3545" if overall_status == "FAILED" else "#009b65"

html_body = f"""<!DOCTYPE html><html><head><meta charset="UTF-8"></head>
<body style="margin:0;padding:18px;background:#edf1f7;font-family:Arial,Helvetica,sans-serif;">
<table width="100%" cellpadding="0" cellspacing="0" style="max-width:1250px;margin:auto;background:#fff;border-collapse:collapse;">
<tr><td style="background:#25265f;padding:35px;text-align:center;color:#fff;">
<h1 style="margin:0;font-size:27px;">Pipeline Execution Report</h1><p style="margin:8px 0 0;">{esc(pipeline_name)}</p></td></tr>
<tr><td style="padding:24px;">
<div style="background:{status_bg};border:1px solid {status_color};padding:18px;margin-bottom:26px;">
<div style="font-size:11px;font-weight:bold;color:{status_color};">OVERALL EXECUTION STATUS</div>
<div style="font-size:23px;font-weight:bold;color:{status_color};">{esc(overall_status)}</div>
<div style="font-size:12px;">Total duration: {esc(duration_text)}</div></div>
{section("Execution Overview","Pipeline identification and execution timeline")}
<table style="{table_css}">{overview_html}</table>
{section("Execution Performance","At-a-glance processing metrics")}
<table width="100%" cellpadding="0" cellspacing="5"><tr>{kpi_html}</tr></table>
{section("Layer Performance","Bronze, Silver and Gold processing summary")}
<table style="{table_css}"><tr>{''.join(f'<th style="{th_css}">{h}</th>' for h in ["Layer","Total","Success","Failed","Running","Other","Inserted","Updated"])}</tr>{layer_html}</table>
{section("Activity Execution Details","Complete activity-level details for this pipeline run")}
<table style="{table_css}"><tr>{''.join(f'<th style="{th_css}">{h}</th>' for h in ["#","Layer","Task","File","Table","Status","Start Time","End Time","Inserted","Updated","Error Details"])}</tr>{activity_html}</table>
{failed_section}
</td></tr>
<tr><td style="background:#25265f;padding:22px;text-align:center;color:#fff;"><h3 style="margin:0;">Thank You</h3>
<p style="margin:8px 0 0;font-size:12px;">Automated notification | Data Engineering Team For Customer Billing</p></td></tr>
</table></body></html>"""

print(f"Prepared HTML report for {pipeline_name}: {overall_status}, {total_tasks} activities.")

# COMMAND ----------

send_html_mail_fn(subject, html_body)
print("Consolidated HTML pipeline notification sent.")

In [0]:
# Send one consolidated notification for this pipeline run.
# Keep the shared function signature unchanged.
# send_html_mail_fn(subject, html_body)
# print("Consolidated pipeline notification sent.")

In [0]:
# df = spark.table("control.etl_run_log")

# # Display all column names, data types, and nullable settings
# for field in df.schema.fields:
#     print(
#         f"Column: {field.name}, "
#         f"Data Type: {field.dataType.simpleString()}, "
#         f"Nullable: {field.nullable}"
#     )
# spark.sql("SHOW SCHEMAS").show(truncate=False)

# spark.sql("SHOW TABLES IN control").show(truncate=False)